In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, mean_squared_error

# ==========================================
# 0. CONFIG: CHANGE YOUR FILE PATHS HERE
# ==========================================
TRAIN_PATH = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv'
TEST_PATH = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv'
SUBMISSION_PATH = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv'

print("--- Step 1: Loading Datasets ---")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SUBMISSION_PATH)

print(f"Train Shape: {train_df.shape} | Test Shape: {test_df.shape}")

# ==========================================
# 1. AUTO-DETECT TARGET AND ID COLUMNS
# ==========================================
potential_targets = list(set(train_df.columns) - set(test_df.columns))
if len(potential_targets) == 1:
    TARGET_COL = potential_targets[0]
else:
    common_names = ['label', 'target', 'class', 'output', 'SalePrice']
    found = [c for c in common_names if c in train_df.columns]
    TARGET_COL = found[0] if found else train_df.columns[-1]

id_candidates = [c for c in ['id', 'ID', 'Id', 'PassengerId'] if c in test_df.columns]
ID_COL = id_candidates[0] if id_candidates else test_df.columns[0]

print(f"-> Auto-detected Target Column: '{TARGET_COL}'")
print(f"-> Auto-detected ID Column: '{ID_COL}'")

# Drop rows where target itself is missing in train_df
train_df = train_df.dropna(subset=[TARGET_COL]).reset_index(drop=True)

# ==========================================
# 2. AUTO-DETECT PROBLEM TYPE
# ==========================================
y_sample = train_df[TARGET_COL]
if y_sample.dtype == 'object' or str(y_sample.dtype) in ['category', 'bool']:
    TASK_TYPE = 'binary_classification' if y_sample.nunique() == 2 else 'multiclass_classification'
elif y_sample.dtype in [float, 'float64'] or y_sample.nunique() > 20:
    TASK_TYPE = 'regression'
elif y_sample.nunique() == 2:
    TASK_TYPE = 'binary_classification'
else:
    TASK_TYPE = 'multiclass_classification'

print(f"-> Auto-detected Task Type: {TASK_TYPE.upper().replace('_', ' ')}")

# ==========================================
# 3. AUTOMATED DATA PREPROCESSING
# ==========================================
print("\n--- Step 2: Preprocessing Data ---")
features = [col for col in train_df.columns if col not in [ID_COL, TARGET_COL] and col in test_df.columns]

X = train_df[features].copy()
y = train_df[TARGET_COL].copy()
X_test = test_df[features].copy()

# Encode target if classification
target_encoder = None
if TASK_TYPE != 'regression':
    target_encoder = LabelEncoder()
    y = pd.Series(target_encoder.fit_transform(y))

# Handle missing values & categorical/boolean variables
for col in X.columns:
    if X[col].dtype == 'object' or str(X[col].dtype) in ['category', 'bool']:
        X[col] = X[col].astype(str)
        X_test[col] = X_test[col].astype(str)
        le = LabelEncoder()
        le.fit(pd.concat([X[col], X_test[col]]))
        X[col] = le.transform(X[col])
        X_test[col] = le.transform(X_test[col])
    else:
        median_val = X[col].median()
        if pd.isna(median_val):
            median_val = 0
        X[col] = X[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)

# Safe Train-Validation Split
can_stratify = (TASK_TYPE != 'regression') and (y.value_counts().min() >= 2)
stratify_target = y if can_stratify else None

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=stratify_target
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 4. TRAINING SYLLABUS-APPROVED MODELS ONLY
# ==========================================
print(f"\n--- Step 3: Training Syllabus-Approved Models ---")

results = []
trained_models = {}

if TASK_TYPE == 'regression':
    from sklearn.linear_model import LinearRegression, Ridge
    from sklearn.neighbors import KNeighborsRegressor
    from sklearn.tree import DecisionTreeRegressor
    from sklearn.svm import SVR
    from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, GradientBoostingRegressor
    
    models = {
        "Linear Regression": (LinearRegression(), True),
        "Ridge Regression": (Ridge(alpha=1.0), True),
        "K-Nearest Neighbors Regressor": (KNeighborsRegressor(n_neighbors=5), True),
        "Decision Tree Regressor": (DecisionTreeRegressor(random_state=42), False),
        "Support Vector Regressor - SVR": (SVR(), True),
        "Random Forest Regressor": (RandomForestRegressor(n_estimators=100, random_state=42), False),
        "Bagging Regressor": (BaggingRegressor(random_state=42), False),
        "Gradient Boosting Regressor": (GradientBoostingRegressor(random_state=42), False)
    }
    
    best_score = float('inf')
    for name, (model, use_scaled) in models.items():
        if use_scaled:
            model.fit(X_train_scaled, y_train)
            preds = model.predict(X_val_scaled)
        else:
            model.fit(X_train, y_train)
            preds = model.predict(X_val)
            
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        results.append((name, rmse, use_scaled))
        trained_models[name] = model
        print(f"[{name}] Validation RMSE: {rmse:.4f}")
        
        if rmse < best_score:
            best_score = rmse
            best_model_name = name

else:
    from sklearn.linear_model import LogisticRegression
    from sklearn.naive_bayes import GaussianNB
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.svm import SVC
    from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, GradientBoostingClassifier
    
    models = {
        "Logistic Regression": (LogisticRegression(max_iter=1000, random_state=42), True),
        "Naïve Bayes": (GaussianNB(), True),
        "K-Nearest Neighbors Classifier": (KNeighborsClassifier(n_neighbors=5), True),
        "Decision Tree Classifier": (DecisionTreeClassifier(random_state=42), False),
        "Support Vector Classifier - SVC": (SVC(probability=True, random_state=42), True),
        "Random Forest Classifier": (RandomForestClassifier(n_estimators=100, random_state=42), False),
        "Bagging Classifier": (BaggingClassifier(random_state=42), False),
        "Gradient Boosting Classifier": (GradientBoostingClassifier(random_state=42), False)
    }
    
    best_score = -1.0
    for name, (model, use_scaled) in models.items():
        if use_scaled:
            model.fit(X_train_scaled, y_train)
            preds = model.predict(X_val_scaled)
        else:
            model.fit(X_train, y_train)
            preds = model.predict(X_val)
            
        acc = accuracy_score(y_val, preds)
        results.append((name, acc, use_scaled))
        trained_models[name] = model
        print(f"[{name}] Validation Accuracy: {acc:.4f}")
        
        if acc > best_score:
            best_score = acc
            best_model_name = name

print(f"\n🏆 Best Syllabus Model: {best_model_name} with Score: {best_score:.4f}")

# ==========================================
# 5. GENERATE SUBMISSION FILE AUTOMATICALLY
# ==========================================
print("\n--- Step 4: Generating Submission File ---")
best_model, use_scaled_flag = trained_models[best_model_name], [m[2] for m in results if m[0] == best_model_name][0]

if use_scaled_flag:
    final_preds = best_model.predict(X_test_scaled)
else:
    final_preds = best_model.predict(X_test)

# Decode predictions back to original labels (e.g., True/False or text classes) if classification
if target_encoder is not None:
    final_preds = target_encoder.inverse_transform(final_preds)

# Match exact column names from sample_submission.csv if available
sub_id_col = sample_sub.columns[0] if len(sample_sub.columns) >= 2 else ID_COL
sub_target_col = sample_sub.columns[1] if len(sample_sub.columns) >= 2 else TARGET_COL

submission = pd.DataFrame({
    sub_id_col: test_df[ID_COL],
    sub_target_col: final_preds
})

submission.to_csv('submission.csv', index=False)

print(f"Success! 'submission.csv' generated with {len(submission)} rows and verified.")
display(submission.head())

# hii 

--- Step 1: Loading Datasets ---


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv'

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, mean_squared_error

# ==========================================
# 0. CONFIG: CHANGE YOUR FILE PATHS HERE
# ==========================================
TRAIN_PATH = '/kaggle/input/competitions/mle-practice-test/train.csv'

TEST_PATH = '/kaggle/input/competitions/mle-practice-test/test.csv'

SUBMISSION_PATH = '/kaggle/input/competitions/mle-practice-test/sample_submission.csv'


print("--- Step 1: Loading Datasets ---")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
submission = pd.read_csv(SUBMISSION_PATH)

print(f"Train Shape: {train_df.shape} | Test Shape: {test_df.shape}")

# ==========================================
# 1. AUTO-DETECT TARGET AND ID COLUMNS
# ==========================================
potential_targets = list(set(train_df.columns) - set(test_df.columns))
if len(potential_targets) == 1:
    TARGET_COL = potential_targets[0]
else:
    common_names = ['label', 'target', 'class', 'output', 'SalePrice']
    found = [c for c in common_names if c in train_df.columns]
    TARGET_COL = found[0] if found else train_df.columns[-1]

id_candidates = [c for c in ['id', 'ID', 'Id', 'PassengerId'] if c in test_df.columns]
ID_COL = id_candidates[0] if id_candidates else test_df.columns[0]

print(f"-> Auto-detected Target Column: '{TARGET_COL}'")
print(f"-> Auto-detected ID Column: '{ID_COL}'")

# ==========================================
# 2. AUTO-DETECT PROBLEM TYPE
# ==========================================
y_sample = train_df[TARGET_COL]
if y_sample.dtype in [float, 'float64'] or y_sample.nunique() > 20:
    TASK_TYPE = 'regression'
elif y_sample.nunique() == 2:
    TASK_TYPE = 'binary_classification'
else:
    TASK_TYPE = 'multiclass_classification'

print(f"-> Auto-detected Task Type: {TASK_TYPE.upper().replace('_', ' ')}")

# ==========================================
# 3. AUTOMATED DATA PREPROCESSING
# ==========================================
print("\n--- Step 2: Preprocessing Data ---")
features = [col for col in train_df.columns if col not in [ID_COL, TARGET_COL]]

X = train_df[features].copy()
y = train_df[TARGET_COL].copy()
X_test = test_df[features].copy()

# Handle missing values & categorical text variables
for col in X.columns:
    if X[col].dtype == 'object' or str(X[col].dtype) == 'category':
        X[col] = X[col].astype(str)
        X_test[col] = X_test[col].astype(str)
        le = LabelEncoder()
        le.fit(pd.concat([X[col], X_test[col]]))
        X[col] = le.transform(X[col])
        X_test[col] = le.transform(X_test[col])
    else:
        median_val = X[col].median()
        X[col].fillna(median_val, inplace=True)
        X_test[col].fillna(median_val, inplace=True)

# Train-Validation Split
stratify_target = y if TASK_TYPE != 'regression' else None
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=stratify_target
)

# Feature Scaling (Needed for Linear, Ridge, Logistic, Naive Bayes, KNN, SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 4. TRAINING SYLLABUS-APPROVED MODELS ONLY
# ==========================================
print(f"\n--- Step 3: Training Syllabus-Approved Models ---")

results = []
trained_models = {}

if TASK_TYPE == 'regression':
    from sklearn.linear_model import LinearRegression, Ridge
    from sklearn.neighbors import KNeighborsRegressor
    from sklearn.tree import DecisionTreeRegressor
    from sklearn.svm import SVR
    from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, GradientBoostingRegressor
    
    models = {
        "Linear Regression (Unit 2)": (LinearRegression(), True),
        "Ridge Regression (Unit 2)": (Ridge(alpha=1.0), True),
        "K-Nearest Neighbors Regressor (Unit 3)": (KNeighborsRegressor(n_neighbors=5), True),
        "Decision Tree Regressor (Unit 3)": (DecisionTreeRegressor(random_state=42), False),
        "Support Vector Regressor - SVR (Unit 3)": (SVR(), True),
        "Random Forest Regressor (Unit 3)": (RandomForestRegressor(n_estimators=100, random_state=42), False),
        "Bagging Regressor (Unit 3)": (BaggingRegressor(random_state=42), False),
        "Gradient Boosting Regressor (Unit 3)": (GradientBoostingRegressor(random_state=42), False)
    }
    
    best_score = float('inf')
    for name, (model, use_scaled) in models.items():
        if use_scaled:
            model.fit(X_train_scaled, y_train)
            preds = model.predict(X_val_scaled)
        else:
            model.fit(X_train, y_train)
            preds = model.predict(X_val)
            
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        results.append((name, rmse, use_scaled))
        trained_models[name] = model
        print(f"[{name}] Validation RMSE: {rmse:.4f}")
        
        if rmse < best_score:
            best_score = rmse
            best_model_name = name

else: # Classification
    from sklearn.linear_model import LogisticRegression
    from sklearn.naive_bayes import GaussianNB
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.svm import SVC
    from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, GradientBoostingClassifier
    
    models = {
        "Logistic Regression": (LogisticRegression(max_iter=1000, random_state=42), True),
        "Naïve Bayes": (GaussianNB(), True),
        "K-Nearest Neighbors Classifier": (KNeighborsClassifier(n_neighbors=5), True),
        "Decision Tree Classifier": (DecisionTreeClassifier(random_state=42), False),
        "Support Vector Classifier - SVC": (SVC(probability=True, random_state=42), True),
        "Random Forest Classifier": (RandomForestClassifier(n_estimators=100, random_state=42), False),
        "Bagging Classifier": (BaggingClassifier(random_state=42), False),
        "Gradient Boosting Classifier": (GradientBoostingClassifier(random_state=42), False)
    }
    
    best_score = -1.0
    for name, (model, use_scaled) in models.items():
        if use_scaled:
            model.fit(X_train_scaled, y_train)
            preds = model.predict(X_val_scaled)
        else:
            model.fit(X_train, y_train)
            preds = model.predict(X_val)
            
        acc = accuracy_score(y_val, preds)
        results.append((name, acc, use_scaled))
        trained_models[name] = model
        print(f"[{name}] Validation Accuracy: {acc:.4f}")
        
        if acc > best_score:
            best_score = acc
            best_model_name = name

print(f"\n🏆 Best Syllabus Model: {best_model_name} with Score: {best_score:.4f}")

# ==========================================
# 5. GENERATE SUBMISSION FILE AUTOMATICALLY
# ==========================================
print("\n--- Step 4: Generating Submission File ---")
best_model, use_scaled_flag = trained_models[best_model_name], [m[2] for m in results if m[0] == best_model_name][0]

if use_scaled_flag:
    final_preds = best_model.predict(X_test_scaled)
else:
    final_preds = best_model.predict(X_test)

# Fix: Build the submission DataFrame directly using test_df IDs to guarantee length match
submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET_COL: final_preds
})

submission.to_csv('submission.csv', index=False)

print(f"Success! 'submission.csv' generated with {len(submission)} rows and verified.")
display(submission.head())